# Use case 3: statistical analysis of the TakeOff public API in a notebook

**Who would do this:** a data-analysis student or a tourism researcher who wants to know whether
cheaper destinations attract more community trips.

**What it does:** loads the public API straight into a pandas DataFrame (no login, no database
access), describes the numbers, measures the correlation between cost and popularity, groups
destinations into cost tiers, and saves a CSV that Excel or Google Sheets can open.

In [ ]:
import pandas as pd

API_URL = "https://suriya.pythonanywhere.com/api/insights/destinations/"

df = pd.read_json(API_URL)   # the API returns a plain JSON array, so pandas reads it directly
print(f"Loaded {len(df)} destinations from {API_URL}")
df

## 1. Descriptive statistics

In [ ]:
df[["public_trips", "avg_daily_cost_usd"]].describe().round(2)

## 2. Is popularity linked to cost?
Pearson measures a straight-line relationship; Spearman uses ranks, which suits small counts.

In [ ]:
pearson = df["avg_daily_cost_usd"].corr(df["public_trips"], method="pearson")
spearman = df["avg_daily_cost_usd"].corr(df["public_trips"], method="spearman")
print(f"Pearson r  (cost vs trips): {pearson:+.3f}")
print(f"Spearman ρ (cost vs trips): {spearman:+.3f}")
direction = "cheaper" if spearman < 0 else "pricier"
strength = "weak" if abs(spearman) < 0.3 else "moderate" if abs(spearman) < 0.6 else "strong"
print(f"Reading: a {strength} tendency for {direction} destinations to get more community trips.")

## 3. Cost tiers

In [ ]:
df["cost_tier"] = pd.cut(df["avg_daily_cost_usd"], bins=[0, 110, 150, float("inf")],
                         labels=["Budget (< $110)", "Mid ($110-150)", "Premium (> $150)"])
tiers = (df.groupby("cost_tier", observed=True)
           .agg(destinations=("destination", "count"),
                community_trips=("public_trips", "sum"),
                avg_daily_cost=("avg_daily_cost_usd", "mean"))
           .round(2))
tiers["trips_per_destination"] = (tiers["community_trips"] / tiers["destinations"]).round(2)
tiers

## 4. Share of community trips by country

In [ ]:
by_country = (df.groupby("country")["public_trips"].sum()
                .sort_values(ascending=False)
                .to_frame("community_trips"))
by_country["share"] = (by_country["community_trips"] / by_country["community_trips"].sum() * 100).round(1).astype(str) + "%"
by_country

## 5. Hand the data to Excel / Google Sheets

In [ ]:
df.to_csv("takeoff_destinations.csv", index=False)
print("Saved takeoff_destinations.csv: open it in Excel or upload it to Google Sheets.")